# Training Scheduler Demo
## Notebook 3: Train with O'Clock

Notebook ini mendemonstrasikan Training Scheduler
untuk otomasi training model ML pada waktu tertentu.

In [ ]:
import sys
import os
import json
import numpy as np

sys.path.insert(0, os.path.join(os.getcwd(), '..'))

print("NumPy version:", np.__version__)
print("TensorFlow version:", __import__('tensorflow').__version__)
print("Ready!")

## 1. Initialize Scheduler

In [ ]:
from src.training_scheduler import TrainingScheduler

scheduler = TrainingScheduler(
    checkpoint_dir='../models/checkpoints',
    output_dir='../outputs'
)

print("Scheduler initialized!")

## 2. Run Training Now

In [ ]:
result = scheduler.run_now(
    training_config={
        'n_samples': 5000,
        'n_features': 20,
        'n_classes': 3,
        'task_type': 'classification',
        'epochs': 15,
        'batch_size': 64,
        'hidden_units': [128, 64, 32],
        'learning_rate': 0.001,
    },
    job_name='demo_training'
)

print(f"\nStatus: {result.get('status')}")
print(f"Time: {result.get('training_time_seconds', 0):.2f}s")
print(f"Epochs: {result.get('epochs_completed')}")
if 'final_metrics' in result:
    for k, v in result['final_metrics'].items():
        print(f"  {k}: {v:.4f}")

## 3. Schedule Training at O'Clock

In [ ]:
# Schedule training at specific times
scheduler.schedule_at_o_clock(hour=6, minute=0, job_name='morning_train')
scheduler.schedule_at_o_clock(hour=12, minute=0, job_name='noon_train')
scheduler.schedule_at_o_clock(hour=18, minute=0, job_name='evening_train')

# Schedule multiple times at once
scheduler.schedule_multiple_o_clock(
    times=['03:00', '09:00', '15:00', '21:00'],
    training_config={'epochs': 5, 'n_samples': 3000}
)

# List all jobs
jobs = scheduler.list_jobs()
print(f"\nTotal scheduled jobs: {len(jobs)}")
for job in jobs:
    print(f"  - {job['name']}: {job}")

## 4. Custom Callbacks

In [ ]:
# Register custom callbacks
def notify_start(job_name, config):
    print(f"\n>>> Notification: Training '{job_name}' starting!")
    print(f">>> Epochs: {config.get('epochs', 'N/A')}")

def notify_complete(job_name, result):
    print(f"\n>>> Notification: Training '{job_name}' done!")
    print(f">>> Status: {result.get('status')}")

scheduler.register_callback('pre_training', notify_start, 'start_notify')
scheduler.register_callback('post_training', notify_complete, 'complete_notify')

# Run with callbacks
result = scheduler.run_now(
    training_config={'epochs': 5, 'n_samples': 2000},
    job_name='callback_demo'
)

## 5. View Training History

In [ ]:
history = scheduler.get_training_history()
print(f"Total training runs: {len(history)}")

for entry in history:
    print(f"\n  Job: {entry.get('job_name')}")
    print(f"  Status: {entry.get('status')}")
    print(f"  Time: {entry.get('training_time_seconds', 0):.2f}s")
    if 'final_metrics' in entry:
        print(f"  Metrics: {entry['final_metrics']}")